## Reading the CSV

In [43]:
import pandas as pd

df = pd.read_csv('cars.csv')
df.head()

,url,price,brand,series,model,year,mileage,transmission,fuel,body,...,engine,hp,drive,condition,heavy_damage,paint_changed,fuel_consumption,fuel_tank,trade_in,seller_type
0,https://www.arabam.com/ilan/galeriden-satilik-...,1.049.000 TL,Ford,Focus,1.5 TDCi Trend X,2018.0,180.000 km,Otomatik,Dizel,Sedan,...,1401 - 1600 cm3,101 - 125 HP,Önden Çekiş,İkinci El,NaN,"1 değişen, 2 boyalı",NaN,NaN,Takasa Uygun,Galeriden
1,https://www.arabam.com/ilan/galeriden-satilik-...,555.750 TL,Ford,Focus,1.6 TDCi Collection,2010.0,357.000 km,Düz,Dizel,Hatchback/5,...,1560 cc,90 hp,Önden Çekiş,İkinci El,Hayır,5 boyalı,"4,7 lt",55 lt,Takasa Uygun,Galeriden
2,https://www.arabam.com/ilan/galeriden-satilik-...,595.000 TL,Citroen,C-Elysée,1.6 HDi Attraction,2014.0,225.000 km,Düz,Dizel,Sedan,...,1560 cc,93 hp,Önden Çekiş,İkinci El,NaN,2 boyalı,"4,3 lt",48 lt,Takasa Uygun,Galeriden
3,https://www.arabam.com/ilan/galeriden-satilik-...,2.059.000 TL,Mercedes - Benz,C,C 180 BlueEFFICIENCY AMG,2014.0,75.000 km,Otomatik,Benzin,Coupe,...,1401 - 1600 cm3,151 - 175 HP,Arkadan İtiş,İkinci El,Belirtilmemiş,Tamamı orjinal,NaN,NaN,Takasa Uygun,Galeriden
4,https://www.arabam.com/ilan/galeriden-satilik-...,1.150.000 TL,Volvo,S60,1.6 D Advance,2014.0,193.000 km,Otomatik,Dizel,Sedan,...,1401 - 1600 cm3,101 - 125 HP,Önden Çekiş,İkinci El,NaN,1 değişen,NaN,NaN,Takasa Uygun,Galeriden


## Cleaning the Data

### We want to clean the price, mileage, and year data to convert them to floats and drop the unrealistic data.

In [44]:
df = df.dropna(subset=["price", "brand", "series", "year", "mileage", "hp"])

In [45]:
df = df.drop_duplicates(subset=["url"])

In [46]:
df["price"] = (
    df["price"]
    .str.replace(" TL", "", regex=False)
    .str.replace(".", "", regex=False)
    .astype(float)
)

df = df[(df["price"] < 20000000) & (df["price"] > 100000)].copy()

df["price"].describe()

count    4.138000e+04
mean     1.106251e+06
std      1.101223e+06
min      1.025000e+05
25%      5.250000e+05
50%      8.550000e+05
75%      1.319962e+06
max      1.998500e+07
Name: price, dtype: float64

In [47]:
df["mileage"] = df["mileage"].str.replace(" km", "", regex=False).str.replace(".", "", regex=False).astype(float)

df = df[
    (df["year"] >= 1980) &
    (df["year"] <= 2026) &
    (df["mileage"] > 0) &
    (df["mileage"] < 1_000_000)
]

df["mileage"].describe()

count     41340.000000
mean     190334.342937
std      103165.567538
min        5080.000000
25%      114894.250000
50%      185000.000000
75%      256000.000000
max      999999.000000
Name: mileage, dtype: float64

In [48]:
df['heavy_damage'] = df['heavy_damage'].fillna('Belirtilmemiş')
df["heavy_damage"].describe()

count             41340
unique                3
top       Belirtilmemiş
freq              26379
Name: heavy_damage, dtype: object

In [49]:
df["hp"] = (
    df["hp"]
    .str.lower()
    .str.replace(" hp", "", regex=False)
    .str.strip()
    .str.split(" - ", expand=True)
    .apply(pd.to_numeric, errors="coerce")
    .astype(float)
    .mean(axis=1)
)
df = df.dropna(subset=["hp"])

In [ ]:
df = df.dropna(subset=["paint_changed"])
df = df[df["paint_changed"] != "Belirtilmemiş"]

df["degisen"] = df["paint_changed"].str.extract(r"(\d+)\s+değişen").fillna(0).astype(int)
df["lokal_boyali"] = df["paint_changed"].str.extract(r"(\d+)\s+lokal\s+boyalı").fillna(0).astype(int)

boyali_only = df["paint_changed"].str.replace(r"\d+\s+lokal\s+boyalı", "", regex=True)
df["boyali"] = boyali_only.str.extract(r"(\d+)\s+boyalı").fillna(0).astype(int)

df.loc[df["paint_changed"] == "Tamamı boyalı", "boyali"] = 12
df.loc[df["paint_changed"] == "Tamamı lokal boyalı", "lokal_boyali"] = 12

In [51]:
df = df.reset_index(drop=True)
df.head()

,url,price,brand,series,model,year,mileage,transmission,fuel,body,...,condition,heavy_damage,paint_changed,fuel_consumption,fuel_tank,trade_in,seller_type,degisen,lokal_boyali,boyali
0,https://www.arabam.com/ilan/galeriden-satilik-...,1049000.0,Ford,Focus,1.5 TDCi Trend X,2018.0,180000.0,Otomatik,Dizel,Sedan,...,İkinci El,Belirtilmemiş,"1 değişen, 2 boyalı",NaN,NaN,Takasa Uygun,Galeriden,1,0,2
1,https://www.arabam.com/ilan/galeriden-satilik-...,555750.0,Ford,Focus,1.6 TDCi Collection,2010.0,357000.0,Düz,Dizel,Hatchback/5,...,İkinci El,Hayır,5 boyalı,"4,7 lt",55 lt,Takasa Uygun,Galeriden,0,0,5
2,https://www.arabam.com/ilan/galeriden-satilik-...,595000.0,Citroen,C-Elysée,1.6 HDi Attraction,2014.0,225000.0,Düz,Dizel,Sedan,...,İkinci El,Belirtilmemiş,2 boyalı,"4,3 lt",48 lt,Takasa Uygun,Galeriden,0,0,2
3,https://www.arabam.com/ilan/galeriden-satilik-...,2059000.0,Mercedes - Benz,C,C 180 BlueEFFICIENCY AMG,2014.0,75000.0,Otomatik,Benzin,Coupe,...,İkinci El,Belirtilmemiş,Tamamı orjinal,NaN,NaN,Takasa Uygun,Galeriden,0,0,0
4,https://www.arabam.com/ilan/galeriden-satilik-...,1150000.0,Volvo,S60,1.6 D Advance,2014.0,193000.0,Otomatik,Dizel,Sedan,...,İkinci El,Belirtilmemiş,1 değişen,NaN,NaN,Takasa Uygun,Galeriden,1,0,0


## Concatenate Brand & Series

### I decided to concatenate these two since each brand's series are special to that brand.

In [52]:
df["brand_series"] = df["brand"] + "_" + df["series"]
df["brand_series"].describe()

count              37872
unique               281
top       Toyota_Corolla
freq                1726
Name: brand_series, dtype: object

## Encoding Brand & Series

### We want to encode these to labels for our model to understand since working with strings does not make sense.

In [53]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["brand_series_encoded"] = le.fit_transform(df["brand_series"])
df["heavy_damage_encoded"] = le.fit_transform(df["heavy_damage"])
print(df["brand_series_encoded"].head())
print(df["heavy_damage_encoded"].head())

0     74
1     74
2     27
3    138
4    270
Name: brand_series_encoded, dtype: int64
0    0
1    2
2    0
3    0
4    0
Name: heavy_damage_encoded, dtype: int64


## Creating the Train & Test Splits

In [54]:
X = df[['brand_series_encoded', 'year', 'mileage', 'heavy_damage_encoded', 'hp', 'degisen', 'boyali', 'lokal_boyali']]
y = df["price"]

print(X.head())
print(y.head())

   brand_series_encoded    year   mileage  heavy_damage_encoded     hp  \
0                    74  2018.0  180000.0                     0  113.0   
1                    74  2010.0  357000.0                     2   90.0   
2                    27  2014.0  225000.0                     0   93.0   
3                   138  2014.0   75000.0                     0  163.0   
4                   270  2014.0  193000.0                     0  113.0   

   degisen  boyali  lokal_boyali  
0        1       2             0  
1        0       5             0  
2        0       2             0  
3        0       0             0  
4        1       0             0  
0    1049000.0
1     555750.0
2     595000.0
3    2059000.0
4    1150000.0
Name: price, dtype: float64


In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

## Training the baseline decision tree

In [56]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print("Decision Tree MAE:", mean_absolute_error(y_test, y_pred))

Decision Tree MAE: 144875.46649064907


## Training the baseline XGBoost

In [57]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print("XGBoost MAE:", mean_absolute_error(y_test, y_pred_xgb))

XGBoost MAE: 103871.62124226485


## Training the baseline LightGBM

In [58]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    objective="regression",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

categorical_features = ["brand_series_encoded", "heavy_damage_encoded"]
lgbm.fit(X_train, y_train, categorical_feature=categorical_features)

y_pred_lgbm = lgbm.predict(X_test)
print("LightGBM MAE:", mean_absolute_error(y_test, y_pred_lgbm))


LightGBM MAE: 93333.85476911397


In [36]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
      "n_estimators": randint(300, 1500),
      "learning_rate": uniform(0.01, 0.1),
      "num_leaves": randint(15, 80),
      "max_depth": [-1, 6, 8, 10, 12],
      "min_child_samples": randint(10, 60),
  }

search = RandomizedSearchCV(
      LGBMRegressor(random_state=42, verbose=-1),
      param_distributions=param_dist,
      n_iter=50,
      scoring="neg_mean_absolute_error",
      cv=3,
      random_state=42,
      n_jobs=-1,
)

search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMRegressor...2, verbose=-1)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': <scipy.stats....t 0x11799d590>, 'max_depth': [-1, 6, ...], 'min_child_samples': <scipy.stats....t 0x11627b9b0>, 'n_estimators': <scipy.stats....t 0x116a988a0>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guid

## Testing the tuned LightGBM on the test set

In [39]:
y_pred = search.best_estimator_.predict(X_test)
print("Tuned LightGBM Test MAE:",
mean_absolute_error(y_test, y_pred))
print("Best params:", search.best_params_)
print("Best CV score:", -search.best_score_)

Tuned LightGBM Test MAE: 134991.54658427398
Best params: {'learning_rate': np.float64(0.013305073290054838), 'max_depth': -1, 'min_child_samples': 10, 'n_estimators': 1453, 'num_leaves': 67}
Best CV score: 132614.7069433751


## Bargain Finder for finding the listings where the listed price is a lot lower than our predicted price

In [60]:
df['predicted_price'] = lgbm.predict(X)
df['discount'] = df['predicted_price'] - df['price']
df['discount_pct'] = df['discount'] / df['predicted_price']

bargains = df[['url', 'brand_series', 'year', 'mileage', 'price', 'predicted_price', 'discount', 'discount_pct', 'heavy_damage', 'paint_changed']]
bargains = bargains.sort_values('discount_pct', ascending=False)
bargains.head(20)

,url,brand_series,year,mileage,price,predicted_price,discount,discount_pct,heavy_damage,paint_changed
37058,https://www.arabam.com/ilan/sahibinden-satilik...,Volvo_S80,1999.0,200000.0,265900.0,1.446756e+06,1.180856e+06,0.816210,Hayır,Tamamı orjinal
37040,https://www.arabam.com/ilan/sahibinden-satilik...,Nissan_Note,2006.0,167500.0,585000.0,2.315031e+06,1.730031e+06,0.747304,Hayır,Tamamı boyalı
35846,https://www.arabam.com/ilan/sahibinden-satilik...,Dacia_Logan,2008.0,370000.0,320000.0,1.170666e+06,8.506658e+05,0.726651,Belirtilmemiş,Tamamı orjinal
13774,https://www.arabam.com/ilan/galeriden-satilik-...,Ford_Taunus,1990.0,150000.0,135000.0,4.895602e+05,3.545602e+05,0.724242,Belirtilmemiş,Tamamı orjinal
4079,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_200,1989.0,100000.0,209000.0,6.777642e+05,4.687642e+05,0.691633,Hayır,10 boyalı
36411,https://www.arabam.com/ilan/sahibinden-satilik...,Volvo_850,1997.0,267000.0,215000.0,6.560023e+05,4.410023e+05,0.672257,Hayır,Tamamı boyalı
37265,https://www.arabam.com/ilan/sahibinden-satilik...,Volvo_850,1992.0,400000.0,150000.0,4.567689e+05,3.067689e+05,0.671606,Hayır,1 boyalı
26111,https://www.arabam.com/ilan/sahibinden-satilik...,Citroen_Xantia,2000.0,235000.0,150000.0,4.312466e+05,2.812466e+05,0.652171,Hayır,Tamamı orjinal
28831,https://www.arabam.com/ilan/sahibinden-satilik...,Seat_Ibiza,1994.0,178000.0,115000.0,3.280800e+05,2.130800e+05,0.649476,Belirtilmemiş,Tamamı orjinal
23774,https://www.arabam.com/ilan/sahibinden-satilik...,Citroen_C5,2011.0,500000.0,154000.0,4.284671e+05,2.744671e+05,0.640579,Hayır,"5 boyalı, 1 lokal boyalı"
